# Notebook 7 – Selecting & Filtering Data
## Introduction

Data selection and filtering are important steps in data analysis.

They help us:

- Select specific rows and columns
- Filter data based on conditions
- Find records that match multiple values
- Select values within a range
- Sort data based on one or more columns

In this notebook, we will use Pandas to perform different data selection and filtering operations.

## Creating a Customer Dataset

We will create a customer dataset containing:

- Customer ID
- Customer Name
- Region
- Membership
- Total Orders
- Total Spend
- Last Order Value
- Churn Risk

This dataset represents customer information that can be used for real-world business analysis.

We will use this dataset to demonstrate different Pandas data selection and filtering techniques such as `loc`, `iloc`, Boolean Indexing, `query()`, `isin()`, `between()`, and sorting.

In [1]:
import pandas as pd
import numpy as np
data = {
    "customer_id":   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    "customer_name": ["Aarav", "Priya", "Rahul", "Sneha", "Vikram",
                       "Ananya", "Karthik", "Divya", "Manoj", "Lakshmi"],
    "region":        ["South", "North", "South", "West", "East",
                       "South", "North", "West", "East", "South"],
    "membership":    ["Gold", "Silver", "Gold", "Bronze", "Silver",
                       "Gold", "Bronze", "Gold", "Silver", "Bronze"],
    "total_orders":  [42, 15, 30, 5, 22, 60, 8, 35, 18, 3],
    "total_spend":   [125000, 32000, 98000, 8000, 45000,
                       210000, 12000, 87000, 39000, 4500],
    "last_order_value": [3200, 1500, 4200, 900, 2100,
                          5000, 1100, 3900, 1800, 700],
    "churn_risk":    ["Low", "Medium", "Low", "High", "Medium",
                        "Low", "High", "Low", "Medium", "High"]
}
df = pd.DataFrame(data)
df

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
5,106,Ananya,South,Gold,60,210000,5000,Low
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


## 1. `loc` — Label-Based Selection

### Concept Explanation
`loc` selects rows and columns using **labels** (row index labels and column names), not integer positions.
Syntax: `df.loc[row_labels, column_labels]`. It's inclusive of both endpoints when slicing.

### Business + AI/ML Example
Our business team wants to look up **customer 103's** name, region, and total spend directly by their row label. A data scientist preparing a churn model might similarly use `loc` to pull specific labeled feature columns for a set of known customers.

In [2]:
df.loc[2, ["customer_name", "region", "total_spend"]]

customer_name    Rahul
region           South
total_spend      98000
Name: 2, dtype: object

**Output Explanation:** Row label `2` corresponds to customer *Rahul*. `loc` returned only the three requested columns for that row, as a Pandas Series (label: value pairs) — useful when the business team wants a quick customer profile lookup.

In [3]:
df.loc[0:4, "customer_name":"membership"]

,customer_name,region,membership
0,Aarav,South,Gold
1,Priya,North,Silver
2,Rahul,South,Gold
3,Sneha,West,Bronze
4,Vikram,East,Silver


**Output Explanation:** `loc[0:4, ...]` returned rows with labels 0 through **4 inclusive** (5 rows) — this is a key difference from Python slicing. The column slice `"customer_name":"membership"` returned every column between (and including) those two column names.

## 2. `iloc` — Position-Based Selection

### Concept Explanation
`iloc` selects rows and columns using **integer positions** (0-based), like standard Python list/array indexing. The end position in a slice is **exclusive**, matching normal Python behaviour.

### Business + AI/ML Example
The analytics team wants the **top 3 rows** of the raw export (regardless of what the index labels are) to spot-check the data. An ML engineer commonly uses `iloc` when slicing feature matrices (`X`) by position after converting a DataFrame to arrays.

In [4]:
df.iloc[0:3, 0:4]

,customer_id,customer_name,region,membership
0,101,Aarav,South,Gold
1,102,Priya,North,Silver
2,103,Rahul,South,Gold


**Output Explanation:** `iloc[0:3, 0:4]` returned rows at positions 0, 1, 2 (3 rows, end-exclusive) and columns at positions 0-3 (`customer_id`, `customer_name`, `region`, `membership`). This is ideal for quick structural checks like "show me the first few rows and columns" without relying on label names.

In [5]:
df.iloc[5, 5]

np.int64(210000)

**Output Explanation:** This returned the single scalar value at row-position 5, column-position 5 — the `total_spend` of the 6th customer (Ananya), i.e., `210000`. Position-based access like this is common when looping through arrays in ML pipelines.

## 3. Boolean Indexing

### Concept Explanation
Boolean indexing selects rows where a **condition evaluates to True**. You build a boolean Series (a "mask") from a condition on a column, then pass that mask into `df[...]`.

### Business + AI/ML Example
The business wants to identify **high-value customers** (`total_spend > 100000`) to send an exclusive loyalty offer. In ML terms, this is identical to filtering a training set to a specific segment before building a segment-specific churn model.

In [6]:
mask = df["total_spend"] > 100000
mask

0     True
1    False
2    False
3    False
4    False
5     True
6    False
7    False
8    False
9    False
Name: total_spend, dtype: bool

In [7]:
high_value_customers = df[mask]
high_value_customers

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
5,106,Ananya,South,Gold,60,210000,5000,Low


**Output Explanation:** The mask marked `True` for rows where `total_spend` exceeds 100,000. Applying it to `df` returned only those 2 customers (*Aarav* and *Ananya*) — exactly the segment the marketing team would target for a premium loyalty campaign.

## 4. Filtering Rows (Combined Conditions)

### Concept Explanation
Multiple boolean conditions can be combined using `&` (AND), `|` (OR), and `~` (NOT). **Each condition must be wrapped in parentheses** because of Python operator precedence.

### Business + AI/ML Example
The business wants **Gold members in the South region** — a segment for a regional VIP event. A data scientist might filter the same way to build a training subset for a region-specific churn model (e.g., "South region churn patterns differ from North").

In [8]:
south_gold = df[(df["region"] == "South") & (df["membership"] == "Gold")]
south_gold

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
0,101,Aarav,South,Gold,42,125000,3200,Low
2,103,Rahul,South,Gold,30,98000,4200,Low
5,106,Ananya,South,Gold,60,210000,5000,Low


**Output Explanation:** Only rows satisfying **both** conditions (South region AND Gold membership) were returned — 3 customers. This matches the business need for a tightly-targeted VIP invite list, and would similarly narrow an ML training set to a specific customer segment.

In [9]:
not_south_or_low_orders = df[~(df["region"] == "South") | (df["total_orders"] < 10)]
not_south_or_low_orders

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
1,102,Priya,North,Silver,15,32000,1500,Medium
3,104,Sneha,West,Bronze,5,8000,900,High
4,105,Vikram,East,Silver,22,45000,2100,Medium
6,107,Karthik,North,Bronze,8,12000,1100,High
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium
9,110,Lakshmi,South,Bronze,3,4500,700,High


**Output Explanation:** This returned every row that is **either** not from the South region **or** has fewer than 10 total orders (or both). It illustrates how `~` negates a condition and `|` combines conditions where at least one must be true — handy for building "at-risk / low-engagement" filters.

## 5. `query()`

### Concept Explanation
`query()` lets you filter rows using a **string expression** instead of boolean masks — often more readable, especially with multiple conditions. Column names are referenced directly as variables inside the string.

### Business + AI/ML Example
The business analyst (who may be more comfortable with SQL-like syntax) wants **customers with more than 20 orders and Medium or High churn risk** — a group that's valuable but at risk of leaving, prime for a retention campaign. This is precisely the kind of filter used to build a labeled dataset for churn model evaluation.

In [10]:
at_risk_valuable = df.query("total_orders > 20 and (churn_risk == 'Medium' or churn_risk == 'High')")
at_risk_valuable

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
4,105,Vikram,East,Silver,22,45000,2100,Medium


**Output Explanation:** `query()` parsed the string expression and returned exactly **1 customer** — *Vikram* — who has more than 20 orders (22) **and** a churn risk of "Medium". Everyone else either had ≤20 orders or a "Low"/other risk that didn't match. The readable syntax avoids repeating `df[...]` and `&`/`|` symbols, which scales well for complex business rules.

## 6. `isin()`

### Concept Explanation
`isin()` checks whether each value in a column belongs to a **given list/set of values** — a clean alternative to chaining many `==` conditions with `|`.

### Business + AI/ML Example
The business wants customers from **North or East regions** for a regional expansion campaign in those two areas. In ML feature engineering, `isin()` is frequently used to flag whether a categorical feature belongs to a chosen subset of categories (e.g., grouping regions into "expansion zone" vs. not).

In [11]:
target_regions = df[df["region"].isin(["North", "East"])]
target_regions

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
1,102,Priya,North,Silver,15,32000,1500,Medium
4,105,Vikram,East,Silver,22,45000,2100,Medium
6,107,Karthik,North,Bronze,8,12000,1100,High
8,109,Manoj,East,Silver,18,39000,1800,Medium


**Output Explanation:** `isin(["North", "East"])` returned `True` for any row whose `region` matches either value in the list, giving us 4 customers. This is far more concise than writing `(df.region=="North") | (df.region=="East")`, and scales cleanly to long lists of categories.

## 7. `between()`

### Concept Explanation
`between(left, right)` returns `True` for values that fall **within a numeric (or date) range**, inclusive by default. It's a shorthand for `(series >= left) & (series <= right)`.

### Business + AI/ML Example
The business wants **mid-tier spenders** — customers whose `total_spend` is between 30,000 and 100,000 — to target with a "step up to Gold" upsell offer. In ML, `between()` is often used to bucket a continuous feature (like spend) into a range for a specific model segment or to flag borderline cases.

In [12]:
mid_tier_spenders = df[df["total_spend"].between(30000, 100000)]
mid_tier_spenders

,customer_id,customer_name,region,membership,total_orders,total_spend,last_order_value,churn_risk
1,102,Priya,North,Silver,15,32000,1500,Medium
2,103,Rahul,South,Gold,30,98000,4200,Low
4,105,Vikram,East,Silver,22,45000,2100,Medium
7,108,Divya,West,Gold,35,87000,3900,Low
8,109,Manoj,East,Silver,18,39000,1800,Medium


**Output Explanation:** `between(30000, 100000)` selected every customer whose spend lies within that inclusive range — 4 customers. This is a cleaner, more readable alternative to writing two separate comparison conditions joined by `&`.

## 8. Sorting Data

### Concept Explanation
`sort_values()` sorts a DataFrame by one or more columns (ascending by default; use `ascending=False` for descending). `sort_index()` sorts by the row index instead. You can sort by multiple columns by passing a list.

### Business + AI/ML Example
The business wants a **leaderboard of customers ranked by total spend**, descending, to prioritize account-manager outreach. Ranking/sorting by a target-correlated feature is also a common exploratory step before training a churn model, to spot patterns at the extremes.

In [13]:
df_sorted = df.sort_values(by="total_spend", ascending=False)
df_sorted[["customer_name", "region", "membership", "total_spend", "churn_risk"]]

,customer_name,region,membership,total_spend,churn_risk
5,Ananya,South,Gold,210000,Low
0,Aarav,South,Gold,125000,Low
2,Rahul,South,Gold,98000,Low
7,Divya,West,Gold,87000,Low
4,Vikram,East,Silver,45000,Medium
8,Manoj,East,Silver,39000,Medium
1,Priya,North,Silver,32000,Medium
6,Karthik,North,Bronze,12000,High
3,Sneha,West,Bronze,8000,High
9,Lakshmi,South,Bronze,4500,High


**Output Explanation:** Rows are now ordered from highest to lowest `total_spend`. *Ananya* tops the list at 210,000, while *Lakshmi* is lowest at 4,500. This kind of ranked view immediately tells the business who their most valuable customers are — and notice most top spenders have **Low** churn risk, a pattern an ML model would learn from.

In [14]:
df.sort_values(by=["membership", "total_spend"], ascending=[True, False])[
    ["customer_name", "membership", "total_spend"]
]

,customer_name,membership,total_spend
6,Karthik,Bronze,12000
3,Sneha,Bronze,8000
9,Lakshmi,Bronze,4500
5,Ananya,Gold,210000
0,Aarav,Gold,125000
2,Rahul,Gold,98000
7,Divya,Gold,87000
4,Vikram,Silver,45000
8,Manoj,Silver,39000
1,Priya,Silver,32000


**Output Explanation:** Rows are first grouped alphabetically by `membership` (Bronze, Gold, Silver — ascending) and, within each membership group, sorted by `total_spend` from highest to lowest. Multi-column sorting like this is exactly how a business would build a "top spender per tier" report.